# 27. '정상만 학습' — 학습 데이터를 바꿔 최종 모델 방식 다시 평가

최종 모델은 이미 정상만 학습 (정상 = unlabeled 생산 샷, 라벨 없음)
→ **라벨로 확인된 정상 샷(labeled PassOrFail=0)**으로 학습하면 어떤지 비교 (변수·모델·전처리는 동일)

| 방법 | 학습 데이터 | 검증 |
|---|---|---|
| **A_ref** (현재) | unlabeled 정상 기준 5,179샷 (CN7) | labeled 전체 (학습에 안 씀) |
| **B_lab_normal** | labeled 확인된 정상 — 테스트 Run 제외 | Run 하나씩 빼기 (LORO) |
| **C_lab+ref** | B + unlabeled 정상 기준 합침 | LORO |

- B·C는 불량 샷을 빼는 데 라벨을 쓰므로 완전 비지도는 아님
- Run마다 모델이 달라 점수는 '학습 정상 대비 백분위'로 통일, 경보 q도 각 fold 학습 정상 기준

In [1]:
import os
import pandas as pd
from normal_only import prepare, run_methods, evaluate, eval_frame1
pd.set_option('display.width', 250); pd.set_option('display.max_columns', 40)
OUT = './outputs/normal_only'
os.makedirs(OUT, exist_ok=True)

R, LOGS, F1 = [], [], []
for p in ['cn7', 'rg3']:
    lab, ref, base = prepare(p)
    out, log = run_methods(p, lab, ref, base)
    R.append(evaluate(p, out)); LOGS.append(log); F1.append(eval_frame1(p, out))
    out.to_csv(f'{OUT}/{p}_scores.csv', index=False, encoding='utf-8-sig')
R = pd.concat(R, ignore_index=True); LOGS = pd.concat(LOGS, ignore_index=True)
R.to_csv(f'{OUT}/summary.csv', index=False, encoding='utf-8-sig')
display(LOGS.pivot_table(index=['제품', '빼둔 Run'], columns='방법', values='학습 샷'))

방법          B_lab_normal  C_lab+ref
제품  빼둔 Run                         
cn7 cn7_R1        1785.0     6964.0
    cn7_R2        1541.0     6720.0
    cn7_R3        1650.0     6829.0
    cn7_R4        1770.0     6949.0
    cn7_R5        1552.0     6731.0
    cn7_R6        1497.0     6676.0
rg3 rg3_R1         554.0     4913.0
    rg3_R2         497.0     4856.0
    rg3_R3         408.0     4767.0
    rg3_R4         525.0     4884.0
    rg3_R5         425.0     4784.0
    rg3_R6         566.0     4925.0
    rg3_R7         595.0     4954.0
    rg3_R8         595.0     4954.0

## 1. ② 전체·샷·초기허용 제외

In [2]:
for p in ['cn7', 'rg3']:
    print(p.upper()); display(R[R['제품'] == p].dropna(axis=1, how='all').round(3))

CN7


,제품,방법,AP,AP_lo,AP_hi,AP/불량률,AP_가스,AP_미성형,Run별 상위10%,전체 상위10%,cn7_R1 내부 AP,q0.99 경보,q0.99 적중,q0.99 정상 중 경보 비율,q0.999 경보,q0.999 적중,q0.999 정상 중 경보 비율,ΔAP(vs 현재),Δ CI,판정
0,cn7,A_ref,0.212,0.011,0.416,27.843,0.007,0.751,4/15,5/15,0.315,29,3,0.013,8,3,0.003,NaN,NaN,NaN
1,cn7,B_lab_normal,0.214,0.013,0.423,28.223,0.009,0.751,5/15,5/15,0.331,24,3,0.011,8,3,0.003,0.003,"[-0.004, 0.014]",차이 없음
2,cn7,C_lab+ref,0.212,0.011,0.417,27.891,0.008,0.751,4/15,5/15,0.318,27,3,0.012,8,3,0.003,0.000,"[-0.001, 0.002]",차이 없음


RG3


,제품,방법,AP,AP_lo,AP_hi,AP/불량률,AP_가스,AP_미성형,Run별 상위10%,전체 상위10%,q0.99 경보,q0.99 적중,q0.99 정상 중 경보 비율,q0.999 경보,q0.999 적중,q0.999 정상 중 경보 비율,ΔAP(vs 현재),Δ CI,판정,rg3_R3 내부 AP
3,rg3,A_ref,0.064,0.051,0.101,1.247,0.050,0.021,3/32,3/32,9,0,0.015,3,0,0.005,NaN,NaN,NaN,0.089
4,rg3,B_lab_normal,0.074,0.053,0.141,1.456,0.064,0.021,6/32,4/32,8,1,0.012,0,0,0.000,0.011,"[-0.001, 0.050]",차이 없음,0.106
5,rg3,C_lab+ref,0.064,0.050,0.100,1.250,0.051,0.020,4/32,3/32,8,0,0.013,3,0,0.005,0.000,"[-0.003, 0.003]",차이 없음,0.089


## 2. ① 공식 2차·행 AP

In [3]:
display(pd.DataFrame(F1).round(3))

,제품,① AP A_ref,① AP B_lab_normal,① AP C_lab+ref
0,cn7,0.373,0.382,0.374
1,rg3,0.024,0.027,0.024


## 해석 가이드
- **B > A (CI상 좋음)** → 확인된 정상으로 학습하는 게 낫다 → 단, 현장에선 라벨된 정상 샷을 따로 모아야 함
- **B ≈ A** → unlabeled 정상 기준으로 충분 (라벨 없이도 같은 성능 = 현장 적용이 쉬움)
- **B < A** → labeled 정상은 수가 적고(약 1,800샷) Run이 몇 개뿐이라 '정상'의 다양성이 부족
- **q 경보의 '정상 중 경보 비율'**이 1−q(1%, 0.1%)에 가까운지 → 임계값 보정 상태